# SentinelGraph — CIC-IDS2017 Exploratory Data Analysis

Notebook: `01_cic_ids2017_eda.ipynb`
Purpose: Perform exploratory data analysis (EDA) on the CIC-IDS2017 dataset and produce reproducible statistics and charts.

## Design constraints (SG-008)

- **All charts and statistics derive from actual loaded data.** No statistics are fabricated.
- **If the dataset is not present**, the notebook runs cleanly, documents the expected location, and skips analysis rather than inventing numbers.
- **Sampling:** For large datasets, rows are sampled with a fixed random seed; the sampling methodology is documented below.
- **Reproducibility:** All random operations use `random_state=42`.

## Expected data location

Downloaded and verified CSVs should live under:

```
data/raw/cic-ids2017/
```

The manifest is expected at `data/manifests/cic-ids2017.json` (created by SG-006). Dataset files are **not** committed to Git.


In [ ]:
import json
import os
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    _HAS_PLOTTING = True
except Exception:  # pragma: no cover - plotting libs optional
    plt = None
    sns = None
    _HAS_PLOTTING = False

plt.style.use("default") if _HAS_PLOTTING else None
sns.set_context("notebook") if _HAS_PLOTTING else None

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print(f"Plotting available: {_HAS_PLOTTING}")

In [ ]:
# Configuration
REPO_ROOT = Path(__file__).resolve().parent.parent.parent if '__file__' in globals() else Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "raw" / "cic-ids2017"
MANIFEST_PATH = REPO_ROOT / "data" / "manifests" / "cic-ids2017.json"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "eda"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Sampling methodology (documented per SG-008):
# If total rows exceed SAMPLE_ROWS, draw a fixed-size random sample.
# Otherwise, use all rows. Seed is fixed for reproducibility.
SAMPLE_ROWS = 200_000
RANDOM_SEED = 42

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Sample threshold: {SAMPLE_ROWS:,} rows | seed: {RANDOM_SEED}")

In [ ]:
# Step 1: Verify dataset availability

manifest_exists = MANIFEST_PATH.is_file()
data_dir_exists = DATA_DIR.is_dir()
csv_files = sorted(DATA_DIR.glob("*.csv")) if data_dir_exists else []

print(f"Manifest exists: {manifest_exists}")
print(f"Data dir exists: {data_dir_exists}")
print(f"CSV files found: {len(csv_files)}")
for f in csv_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

DATA_AVAILABLE = data_dir_exists and len(csv_files) > 0
print(f"\nDATA_AVAILABLE: {DATA_AVAILABLE}")
if not DATA_AVAILABLE:
    print("\nNo CSV data found. Place verified CIC-IDS2017 CSVs under:")
    print(f"  {DATA_DIR}")
    print("\nAnalysis will be skipped; no statistics will be fabricated.")

In [ ]:
# Step 2: Load data (with documented sampling)

df = None
sampled = False
original_row_count = 0

if DATA_AVAILABLE:
    frames = []
    for csv_path in csv_files:
        try:
            chunk = pd.read_csv(csv_path, low_memory=False)
            chunk["_source_file"] = csv_path.name
            frames.append(chunk)
            print(f"Loaded {csv_path.name}: {len(chunk):,} rows x {chunk.shape[1]} cols")
        except Exception as exc:
            print(f"Failed to load {csv_path.name}: {exc}")

    if frames:
        df = pd.concat(frames, ignore_index=True)
        original_row_count = len(df)

        # Sampling: fixed-size random sample when data is large.
        if original_row_count > SAMPLE_ROWS:
            df = df.sample(n=SAMPLE_ROWS, random_state=RANDOM_SEED).reset_index(drop=True)
            sampled = True
            print(f"\nSampled {SAMPLE_ROWS:,} rows from {original_row_count:,} total (seed={RANDOM_SEED}).")
            print("Sampling methodology: simple random sampling without replacement, fixed seed.")
        else:
            print(f"\nUsing all {original_row_count:,} rows (below sample threshold {SAMPLE_ROWS:,}).")

        print(f"\nFinal analysis frame: {len(df):,} rows x {df.shape[1]} cols")
        print(f"Columns: {list(df.columns)}")
    else:
        print("No CSV files could be loaded.")

print(f"\nData loaded: {df is not None} | Sampled: {sampled} | Original rows: {original_row_count:,}")

## 1. Dataset dimensions

Reports total rows, columns, memory usage, and per-file row breakdown when data is available.

In [ ]:
if DATA_AVAILABLE and df is not None:
    print("=== Dataset Dimensions ===")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
    print("\nPer-file row breakdown:")
    if "_source_file" in df.columns:
        print(df["_source_file"].value_counts().sort_index().to_string())
    print("\nColumn dtypes:")
    print(df.dtypes.to_string())
else:
    print("Skipped: no data available.")

## 2. Feature distributions

Histograms for numeric features (sampled to a representative subset for speed).

In [ ]:
if DATA_AVAILABLE and df is not None and _HAS_PLOTTING:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Exclude identifier-like / high-cardinality index columns from histograms.
    exclude_like = ["flow id", "src ip", "dst ip", "timestamp"]
    plot_cols = [c for c in numeric_cols if not any(x in c.lower() for x in exclude_like)][:12]
    
    print(f"Plotting distributions for {len(plot_cols)} numeric features: {plot_cols}")
    
    n_rows = int(np.ceil(len(plot_cols) / 3))
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
    axes = np.atleast_2d(axes)
    for i, col in enumerate(plot_cols):
        ax = axes[i // 3, i % 3]
        series = df[col].dropna()
        # Cap extreme outliers for display only (not for analysis).
        lo, hi = series.quantile([0.001, 0.999])
        display = series[(series >= lo) & (series <= hi)]
        ax.hist(display, bins=40, color="steelblue", edgecolor="black", alpha=0.7)
        ax.set_title(col)
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")
    # Hide unused subplots.
    for j in range(len(plot_cols), n_rows * 3):
        axes[j // 3, j % 3].set_visible(False)
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "feature_distributions.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {ARTIFACT_DIR / 'feature_distributions.png'}")
else:
    print("Skipped: no data or plotting unavailable.")

## 3. Class distribution

Bar chart of the `Label` column (binary benign vs. attack, and multiclass attack categories).

In [ ]:
if DATA_AVAILABLE and df is not None and _HAS_PLOTTING:
    label_col = None
    for candidate in ["Label", "label", "class"]:
        if candidate in df.columns:
            label_col = candidate
            break
    
    if label_col:
        counts = df[label_col].value_counts().sort_values(ascending=False)
        print(f"Class column: {label_col}")
        print(counts.to_string())
        print(f"\nUnique classes: {len(counts)}")
        print(f"Benign rows: {int(counts.get('BENIGN', counts.get('Benign', 0))):,}")
        print(f"Attack rows: {len(df) - int(counts.get('BENIGN', counts.get('Benign', 0))):,}")
        
        fig, ax = plt.subplots(figsize=(10, 5))
        counts.plot(kind="bar", color=["#2ca02c" if str(v).lower() in ("benign",) else "#d62728" for v in counts.index], ax=ax)
        ax.set_title("Class Distribution")
        ax.set_xlabel("Class")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)
        fig.tight_layout()
        fig.savefig(ARTIFACT_DIR / "class_distribution.png", dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {ARTIFACT_DIR / 'class_distribution.png'}")
    else:
        print("No label column found; skipping class distribution.")
else:
    print("Skipped: no data or plotting unavailable.")

## 4. Attack imbalance

Quantifies imbalance between benign and attack classes, and across attack categories (imbalance ratio = majority / minority).

In [ ]:
if DATA_AVAILABLE and df is not None:
    label_col = None
    for candidate in ["Label", "label", "class"]:
        if candidate in df.columns:
            label_col = candidate
            break
    
    if label_col:
        counts = df[label_col].value_counts()
        benign = int(counts.get("BENIGN", counts.get("Benign", 0)))
        attack = len(df) - benign
        imbalance_ratio = (max(benign, attack) / min(benign, attack)) if min(benign, attack) > 0 else float("inf")
        
        print("=== Attack Imbalance ===")
        print(f"Benign: {benign:,}")
        print(f"Attack: {attack:,}")
        print(f"Imbalance ratio (benign:attack): {imbalance_ratio:.2f}:1")
        print(f"Attack share: {attack / len(df) * 100:.2f}%")
        
        # Per-attack-category imbalance (excluding benign).
        attack_counts = counts[[idx for idx in counts.index if str(idx).lower() not in ("benign",)]]
        if len(attack_counts) > 0:
            print("\nPer-attack-category counts:")
            print(attack_counts.sort_values(ascending=False).to_string())
            cat_ratio = (attack_counts.max() / attack_counts.min()) if attack_counts.min() > 0 else float("inf")
            print(f"\nAttack-category imbalance ratio (max:min): {cat_ratio:.2f}:1")
    else:
        print("No label column found; skipping imbalance analysis.")
else:
    print("Skipped: no data available.")

## 5. Missing values

Reports missing-value counts and percentages per column.

In [ ]:
if DATA_AVAILABLE and df is not None:
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_report = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct,
    }).sort_values("missing_count", ascending=False)
    
    print("=== Missing Values ===")
    print(f"Total missing cells: {int(missing.sum()):,}")
    print(f"Columns with missing values: {(missing > 0).sum()} / {df.shape[1]}")
    print("\nMissing-value report (top 20):")
    print(missing_report.head(20).to_string())
    
    if _HAS_PLOTTING and (missing > 0).any():
        plot_missing = missing_report[missing_report["missing_count"] > 0].head(15)
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_missing["missing_count"].plot(kind="barh", color="orange", ax=ax)
        ax.set_title("Missing Values per Column (top 15)")
        ax.set_xlabel("Missing Count")
        fig.tight_layout()
        fig.savefig(ARTIFACT_DIR / "missing_values.png", dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {ARTIFACT_DIR / 'missing_values.png'}")
else:
    print("Skipped: no data available.")

## 6. Duplicate rows

Reports exact duplicate rows and duplicate key combinations.

In [ ]:
if DATA_AVAILABLE and df is not None:
    total_dups = int(df.duplicated().sum())
    print("=== Duplicate Rows ===")
    print(f"Exact duplicate rows: {total_dups:,} / {len(df):,} ({total_dups / len(df) * 100:.2f}%)")
    
    # Identifier-based duplicate check if identifier columns exist.
    id_cols = [c for c in ["Flow ID", "Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol", "Timestamp"] if c in df.columns]
    if id_cols:
        key_dups = int(df.duplicated(subset=id_cols).sum())
        print(f"Duplicate key combinations ({', '.join(id_cols)}): {key_dups:,}")
    else:
        print("No identifier columns found for key-duplicate check.")
else:
    print("Skipped: no data available.")

## 7. Infinite values

Reports infinite values (from division-by-zero in feature generation) per numeric column.

In [ ]:
if DATA_AVAILABLE and df is not None:
    numeric_df = df.select_dtypes(include=[np.number])
    inf_counts = np.isinf(numeric_df).sum()
    inf_report = pd.DataFrame({
        "inf_count": inf_counts,
    }).sort_values("inf_count", ascending=False)
    
    print("=== Infinite Values ===")
    print(f"Total infinite cells: {int(inf_counts.sum()):,}")
    print(f"Columns with infinite values: {(inf_counts > 0).sum()} / {numeric_df.shape[1]}")
    if (inf_counts > 0).any():
        print("\nInfinite-value report (top 20):")
        print(inf_report.head(20).to_string())
    else:
        print("No infinite values detected.")
else:
    print("Skipped: no data available.")

## 8. Numerical correlations

Correlation heatmap of numeric features (top correlated pairs listed).

In [ ]:
if DATA_AVAILABLE and df is not None and _HAS_PLOTTING:
    numeric_df = df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).dropna()
    if numeric_df.shape[1] > 1 and len(numeric_df) > 0:
        corr = numeric_df.corr()
        
        print("=== Numerical Correlations ===")
        # Top correlated pairs (absolute value), excluding self-correlation.
        corr_abs = corr.abs()
        upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
        top_pairs = (
            upper.stack()
            .sort_values(ascending=False)
            .head(15)
        )
        print("Top correlated feature pairs (|r|):")
        print(top_pairs.round(3).to_string())
        
        # Heatmap of a manageable subset.
        plot_cols = corr.columns[:25].tolist()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr.loc[plot_cols, plot_cols], cmap="coolwarm", center=0, annot=False, ax=ax)
        ax.set_title("Feature Correlation Heatmap (top 25)")
        fig.tight_layout()
        fig.savefig(ARTIFACT_DIR / "correlation_heatmap.png", dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {ARTIFACT_DIR / 'correlation_heatmap.png'}")
    else:
        print("Insufficient numeric data for correlation analysis.")
else:
    print("Skipped: no data or plotting unavailable.")

## 9. Outliers

Boxplots for selected numeric features and IQR-based outlier counts.

In [ ]:
if DATA_AVAILABLE and df is not None and _HAS_PLOTTING:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude_like = ["flow id", "src ip", "dst ip", "timestamp"]
    plot_cols = [c for c in numeric_cols if not any(x in c.lower() for x in exclude_like)][:6]
    
    print("=== Outliers (IQR method) ===")
    outlier_report = []
    for col in plot_cols:
        series = df[col].dropna()
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((series < lower) | (series > upper)).sum())
        outlier_report.append({"feature": col, "iqr": iqr, "outlier_count": n_out, "outlier_pct": round(n_out / len(series) * 100, 2)})
        print(f"{col}: {n_out:,} outliers ({n_out / len(series) * 100:.2f}%)")
    
    if plot_cols:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        for i, col in enumerate(plot_cols):
            axes[i].boxplot(df[col].dropna(), showfliers=True)
            axes[i].set_title(col)
        for j in range(len(plot_cols), len(axes)):
            axes[j].set_visible(False)
        fig.tight_layout()
        fig.savefig(ARTIFACT_DIR / "outliers.png", dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {ARTIFACT_DIR / 'outliers.png'}")
else:
    print("Skipped: no data or plotting unavailable.")

## 10. Temporal patterns

If a timestamp column exists, examines flow volume over time.

In [ ]:
if DATA_AVAILABLE and df is not None and _HAS_PLOTTING:
    ts_col = None
    for candidate in ["Timestamp", "timestamp", "Flow Start", "flow_start"]:
        if candidate in df.columns:
            ts_col = candidate
            break
    
    if ts_col:
        print(f"=== Temporal Patterns (column: {ts_col}) ===")
        try:
            ts = pd.to_datetime(df[ts_col], errors="coerce")
            print(f"Parsed timestamps: {ts.notna().sum():,} / {len(df):,}")
            print(f"Time range: {ts.min()} to {ts.max()}")
            
            # Hour-of-day flow volume.
            df["_hour"] = ts.dt.hour
            hourly = df["_hour"].value_counts().sort_index()
            print("\nFlows by hour of day:")
            print(hourly.to_string())
            
            fig, ax = plt.subplots(figsize=(10, 4))
            hourly.plot(kind="bar", color="steelblue", ax=ax)
            ax.set_title("Flow Volume by Hour of Day")
            ax.set_xlabel("Hour")
            ax.set_ylabel("Flow Count")
            fig.tight_layout()
            fig.savefig(ARTIFACT_DIR / "temporal_patterns.png", dpi=120, bbox_inches="tight")
            plt.show()
            print(f"Saved: {ARTIFACT_DIR / 'temporal_patterns.png'}")
        except Exception as exc:
            print(f"Temporal analysis failed: {exc}")
    else:
        print("No timestamp column found; skipping temporal analysis.")
else:
    print("Skipped: no data or plotting unavailable.")

## 11. Potential leakage columns

Identifies identifier columns and near-constant or high-cardinality columns that could leak information or add no signal.

In [ ]:
if DATA_AVAILABLE and df is not None:
    print("=== Potential Leakage / Low-Signal Columns ===")
    
    id_keywords = ["flow id", "src ip", "dst ip", "src port", "dst port", "timestamp", "mac"]
    id_cols = [c for c in df.columns if any(k in c.lower() for k in id_keywords)]
    print(f"Identifier-like columns: {id_cols}")
    
    # Near-constant columns (single unique value).
    constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
    print(f"Near-constant columns (<=1 unique): {constant_cols}")
    
    # High-cardinality identifier columns.
    high_card = []
    for c in df.columns:
        if df[c].nunique(dropna=True) > len(df) * 0.9:
            high_card.append(c)
    print(f"High-cardinality columns (>90% unique): {high_card}")
    
    print("\nRecommendation: drop identifier-like and constant columns before training.")
else:
    print("Skipped: no data available.")

## 12. Source and destination communication characteristics

Summarizes unique IPs, port usage, protocols, and bidirectional flow asymmetry when identifier columns are present.

In [ ]:
if DATA_AVAILABLE and df is not None:
    print("=== Source / Destination Communication Characteristics ===")
    
    ip_cols = [c for c in ["Src IP", "Dst IP"] if c in df.columns]
    port_cols = [c for c in ["Src Port", "Dst Port"] if c in df.columns]
    proto_cols = [c for c in ["Protocol", "protocol"] if c in df.columns]
    
    if ip_cols:
        for col in ip_cols:
            n_unique = df[col].nunique(dropna=True)
            print(f"{col}: {n_unique:,} unique values")
    
    if port_cols:
        for col in port_cols:
            top_ports = df[col].value_counts().head(10)
            print(f"\n{col} top 10:")
            print(top_ports.to_string())
    
    if proto_cols:
        for col in proto_cols:
            proto_counts = df[col].value_counts()
            print(f"\n{col} distribution:")
            print(proto_counts.to_string())
    
    # Bidirectional asymmetry: flows where src->dst vs dst->src appear both ways.
    if {"Src IP", "Dst IP"}.issubset(df.columns):
        pairs = df[["Src IP", "Dst IP"]].dropna()
        forward = set(zip(pairs["Src IP"], pairs["Dst IP"]))
        reverse = set(zip(pairs["Dst IP"], pairs["Src IP"]))
        bidirectional = forward & reverse
        print(f"\nUnique directed pairs: {len(forward):,}")
        print(f"Bidirectional pairs (both directions observed): {len(bidirectional):,}")
        print(f"Asymmetry ratio: {len(forward) / max(len(bidirectional), 1):.2f}")
else:
    print("Skipped: no data available.")

## 13. Data quality report

Writes a machine-readable data quality report to `artifacts/eda/data_quality_report.json`.

In [ ]:
report = {
    "dataset": "cic-ids2017",
    "data_available": DATA_AVAILABLE,
    "sampling": {
        "sampled": bool(sampled),
        "sample_rows": SAMPLE_ROWS,
        "random_seed": RANDOM_SEED,
        "original_row_count": original_row_count,
        "analyzed_row_count": int(len(df)) if df is not None else 0,
        "methodology": "simple random sampling without replacement, fixed seed, applied when total rows > sample threshold",
    },
}

if DATA_AVAILABLE and df is not None:
    label_col = None
    for candidate in ["Label", "label", "class"]:
        if candidate in df.columns:
            label_col = candidate
            break
    
    missing = df.isna().sum()
    numeric_df = df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    inf_counts = np.isinf(numeric_df).sum()
    
    report["dimensions"] = {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "memory_mb": float(df.memory_usage(deep=True).sum() / 1024 / 1024),
    }
    report["missing_values"] = {
        "total_missing_cells": int(missing.sum()),
        "columns_with_missing": int((missing > 0).sum()),
    }
    report["duplicates"] = {
        "exact_duplicate_rows": int(df.duplicated().sum()),
    }
    report["infinite_values"] = {
        "total_inf_cells": int(inf_counts.sum()),
        "columns_with_inf": int((inf_counts > 0).sum()),
    }
    if label_col:
        counts = df[label_col].value_counts()
        benign = int(counts.get("BENIGN", counts.get("Benign", 0)))
        report["class_distribution"] = {str(k): int(v) for k, v in counts.items()}
        report["imbalance"] = {
            "benign": benign,
            "attack": int(len(df) - benign),
        }

report_path = ARTIFACT_DIR / "data_quality_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"Data quality report written to: {report_path}")
print(json.dumps(report, indent=2))

## 14. EDA summary

Documented findings from the analysis above (empty if no data was available).

In [ ]:
print("=== EDA Summary ===")
if not DATA_AVAILABLE:
    print("No CIC-IDS2017 data was available for analysis.")
    print("Place verified CSVs under data/raw/cic-ids2017/ and re-run this notebook.")
    print("No statistics or charts were fabricated.")
else:
    findings = []
    findings.append(f"Analyzed {len(df):,} rows x {df.shape[1]} columns (sampled={sampled}).")
    
    missing = df.isna().sum()
    findings.append(f"Missing cells: {int(missing.sum()):,} across {(missing > 0).sum()} columns.")
    
    inf_counts = np.isinf(df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)).sum()
    findings.append(f"Infinite cells: {int(inf_counts.sum()):,} across {(inf_counts > 0).sum()} numeric columns.")
    
    findings.append(f"Exact duplicate rows: {int(df.duplicated().sum()):,}.")
    
    label_col = None
    for candidate in ["Label", "label", "class"]:
        if candidate in df.columns:
            label_col = candidate
            break
    if label_col:
        counts = df[label_col].value_counts()
        benign = int(counts.get("BENIGN", counts.get("Benign", 0)))
        attack = len(df) - benign
        findings.append(f"Class split: benign={benign:,}, attack={attack:,} (attack share {attack/len(df)*100:.2f}%).")
    
    for f_ in findings:
        print(f"- {f_}")
    print("\nCharts saved under:", ARTIFACT_DIR)